# Assignment 5 — Catching Data Leakage Before It Catches You
**Feature Engineering & MLOps — Target Leakage & Preprocessing Leakage**

Dataset: `ASSIGNMENTS/data/customer_churn_a5.csv` (600 telecom customers)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42

df = pd.read_csv('/Users/muhammedfaheem/Desktop/AI_DS/Feature Engineering/ASSIGNMENTS/data/customer_churn_a5.csv')
print(df.shape)
df.head()

(600, 8)


,customer_id,tenure_months,monthly_charges,contract_type,support_calls,churn,days_since_cancellation,final_bill_amount
0,20406,13,99.17,Month-to-month,2,0,NaN,NaN
1,20531,24,66.77,Month-to-month,1,0,NaN,NaN
2,20307,15,44.03,One year,1,0,NaN,NaN
3,20391,3,61.08,Two year,3,0,NaN,NaN
4,20147,8,106.72,Two year,1,0,NaN,NaN


## Section 4.1 — Target Leakage: Prove It, Don't Just Assert It

### 4.1.1 — `days_since_cancellation` is only present when `churn == 1`

In [2]:
non_null_days = df['days_since_cancellation'].notna().sum()
churn1_count = (df['churn'] == 1).sum()

print('Rows with non-null days_since_cancellation:', non_null_days)
print('Rows with churn == 1:', churn1_count)
print('Counts are exactly equal:', non_null_days == churn1_count)

Rows with non-null days_since_cancellation: 149
Rows with churn == 1: 149
Counts are exactly equal: True


**Interpretation:** the count matches exactly. `days_since_cancellation` is only ever populated *after* a customer has already cancelled — meaning it is literally a downstream consequence of the label, not a predictor of it. This is a textbook case of target leakage: the feature cannot possibly exist at the moment we'd need to make a prediction.

### 4.1.2 — Correlation of `final_bill_amount` vs. a legitimate feature

In [3]:
df['final_bill_amount_filled'] = df['final_bill_amount'].fillna(-1)

corr_final_bill = df['final_bill_amount_filled'].corr(df['churn'])
corr_tenure = df['tenure_months'].corr(df['churn'])

print('corr(final_bill_amount, churn):', corr_final_bill)
print('corr(tenure_months, churn):     ', corr_tenure)

corr(final_bill_amount, churn): 0.9249777122193823
corr(tenure_months, churn):      -0.220718079977007


**Interpretation:** `final_bill_amount` correlates with churn at **~0.92** — an almost suspiciously perfect relationship for a real-world business feature. Compare that to `tenure_months`, a genuinely legitimate predictor, which correlates at only **~-0.22**. A correlation this close to 1.0 is a red flag: it usually means the "feature" is really just encoding the answer (filling with -1 for non-churners makes the separation even sharper, since -1 vs. a real dollar amount is almost a direct churn/no-churn switch).

### 4.1.3 — Model A: legitimate features only

In [4]:
legit_features = ['tenure_months', 'monthly_charges', 'contract_type', 'support_calls']
X = pd.get_dummies(df[legit_features], columns=['contract_type'], drop_first=True)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

numeric_cols_A = ['tenure_months', 'monthly_charges', 'support_calls']
scaler_A = StandardScaler()
X_train_A, X_test_A = X_train.copy(), X_test.copy()
X_train_A[numeric_cols_A] = scaler_A.fit_transform(X_train[numeric_cols_A])
X_test_A[numeric_cols_A] = scaler_A.transform(X_test[numeric_cols_A])

model_A = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_A.fit(X_train_A, y_train)

acc_A = accuracy_score(y_test, model_A.predict(X_test_A))
auc_A = roc_auc_score(y_test, model_A.predict_proba(X_test_A)[:, 1])

print('Model A test accuracy:', acc_A)
print('Model A test ROC-AUC: ', auc_A)

Model A test accuracy: 0.7666666666666667
Model A test ROC-AUC:  0.8018518518518518


### 4.1.4 — Model B: legitimate features + leaked features

In [5]:
leak_features = legit_features + ['days_since_cancellation', 'final_bill_amount']
Xb = df[leak_features].copy()
Xb['days_since_cancellation'] = Xb['days_since_cancellation'].fillna(-1)
Xb['final_bill_amount'] = Xb['final_bill_amount'].fillna(-1)
Xb = pd.get_dummies(Xb, columns=['contract_type'], drop_first=True)

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

numeric_cols_B = ['tenure_months', 'monthly_charges', 'support_calls',
                   'days_since_cancellation', 'final_bill_amount']
scaler_B = StandardScaler()
Xb_train_s, Xb_test_s = Xb_train.copy(), Xb_test.copy()
Xb_train_s[numeric_cols_B] = scaler_B.fit_transform(Xb_train[numeric_cols_B])
Xb_test_s[numeric_cols_B] = scaler_B.transform(Xb_test[numeric_cols_B])

model_B = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_B.fit(Xb_train_s, yb_train)

acc_B = accuracy_score(yb_test, model_B.predict(Xb_test_s))
auc_B = roc_auc_score(yb_test, model_B.predict_proba(Xb_test_s)[:, 1])

print('Model B test accuracy:', acc_B)
print('Model B test ROC-AUC: ', auc_B)

Model B test accuracy: 1.0
Model B test ROC-AUC:  1.0


### 4.1.5 — The gap, and what happens in production

In [6]:
print('Accuracy gap (B - A):', acc_B - acc_A)
print('ROC-AUC gap (B - A):  ', auc_B - auc_A)

Accuracy gap (B - A): 0.23333333333333328
ROC-AUC gap (B - A):   0.19814814814814818


**Gap:** Model B looks nearly perfect (100% accuracy, 1.0 AUC) versus Model A's honest ~77% accuracy / 0.80 AUC — a jump of roughly 20-23 points that comes entirely from two features that don't exist yet at prediction time.

**Why Model B would fail in production:** `days_since_cancellation` and `final_bill_amount` are only generated *after* a customer has already cancelled — a brand-new applicant, or any current customer we're trying to score today, has no value for either field (they'd arrive as null or some placeholder like -1 for every single row). The model would essentially see "final_bill_amount = -1" for 100% of real scoring requests, collapse onto whatever pattern that placeholder implies, and lose almost all of the discriminative signal it thinks it has — its real-world accuracy would crash back down toward Model A's honest baseline (or worse, since it was never trained to lean on the legitimate features).

## Section 4.2 — Preprocessing Leakage: Wrong Order vs. Right Order

### 4.2.1 — WRONG: scale on the full dataset, then split

In [7]:
numeric_cols = ['tenure_months', 'monthly_charges', 'support_calls']

scaler_wrong = StandardScaler()
X_all_scaled = scaler_wrong.fit_transform(df[numeric_cols])

print('WRONG scaler mean_: ', scaler_wrong.mean_)
print('WRONG scaler scale_:', scaler_wrong.scale_)

WRONG scaler mean_:  [19.56166667 65.92201667  2.165     ]
WRONG scaler scale_: [17.52454081 22.79931919  1.49814608]


### 4.2.2 — RIGHT: split first, then fit scaler on training data only

In [8]:
Xr_train_raw, Xr_test_raw, yr_train, yr_test = train_test_split(
    df[numeric_cols], y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

scaler_right = StandardScaler()
scaler_right.fit(Xr_train_raw)

print('RIGHT scaler mean_: ', scaler_right.mean_)
print('RIGHT scaler scale_:', scaler_right.scale_)

RIGHT scaler mean_:  [19.73958333 65.72710417  2.11458333]
RIGHT scaler scale_: [17.70077964 22.41337332  1.46820321]


### 4.2.3 — Numeric difference, and why the principle matters regardless of size

In [9]:
mean_diff = scaler_wrong.mean_ - scaler_right.mean_
print('Mean difference (wrong - right):', mean_diff)

Mean difference (wrong - right): [-0.17791667  0.1949125   0.05041667]


**Interpretation:** on this dataset the gap between the two scalers' means is small (well under 1 unit for each feature). That's a coincidence of this particular 600-row sample, not a property of the *approach*. The principle matters regardless of the gap's size because fitting on the full dataset means the test set's values have literally leaked into the numbers (mean, standard deviation) used to transform the training data — the model's preprocessing step has "seen" the test distribution before training even starts. On a smaller dataset, a dataset with more outliers, or a test set that isn't a close statistical match to training (which is often exactly the situation in production, where new data drifts from what you trained on), that same leak can produce a much bigger — and misleadingly optimistic — gap in reported performance.

### 4.2.4 — Rebuild as a single sklearn Pipeline

In [10]:
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
])
pipe.fit(Xr_train_raw)

pipeline_mean = pipe.named_steps['scaler'].mean_
pipeline_scale = pipe.named_steps['scaler'].scale_

print('Pipeline scaler mean_: ', pipeline_mean)
print('Pipeline scaler scale_:', pipeline_scale)
print('Matches manual correct version (mean)? ', np.allclose(pipeline_mean, scaler_right.mean_))
print('Matches manual correct version (scale)?', np.allclose(pipeline_scale, scaler_right.scale_))

Pipeline scaler mean_:  [19.73958333 65.72710417  2.11458333]
Pipeline scaler scale_: [17.70077964 22.41337332  1.46820321]
Matches manual correct version (mean)?  True
Matches manual correct version (scale)? True


The `Pipeline`, fit only on the training split, reproduces the manual "correct" scaler's statistics exactly — confirming the pipeline approach is equivalent to doing it by hand, just safer and less error-prone.

## Section 4.3 — The Fix: Final Honest, Deployable Model

In [11]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), numeric_cols_A)
], remainder='passthrough')

final_full_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

X_final = pd.get_dummies(df[legit_features], columns=['contract_type'], drop_first=True)
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_final, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

final_full_pipe.fit(Xf_train, yf_train)

acc_final = accuracy_score(yf_test, final_full_pipe.predict(Xf_test))
auc_final = roc_auc_score(yf_test, final_full_pipe.predict_proba(Xf_test)[:, 1])

print('Final fixed model test accuracy:', acc_final)
print('Final fixed model test ROC-AUC: ', auc_final)
print('Matches Model A accuracy?', np.isclose(acc_final, acc_A))
print('Matches Model A ROC-AUC? ', np.isclose(auc_final, auc_A))

Final fixed model test accuracy: 0.7666666666666667
Final fixed model test ROC-AUC:  0.8018518518518518
Matches Model A accuracy? True
Matches Model A ROC-AUC?  True


The final, properly-ordered pipeline (split → fit preprocessing on train only → train model) reproduces Model A's numbers exactly. This is the honest, deployable version: no target leakage (no post-cancellation features) and no preprocessing leakage (scaler never sees the test set).

## Bonus — Leakage Inside Cross-Validation (+10%)

In [12]:
# WRONG: scale on the whole dataset, then run CV on the already-scaled data
scaler_bonus = StandardScaler()
X_scaled_all = scaler_bonus.fit_transform(df[numeric_cols])

leaked_scores = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    X_scaled_all, y, cv=5, scoring='roc_auc')

print('Leaked CV scores:', leaked_scores)
print('Leaked CV mean:  ', leaked_scores.mean())

Leaked CV scores: [0.6585828  0.66148148 0.72888889 0.68074074 0.66074074]
Leaked CV mean:   0.6780869296731366


In [13]:
# RIGHT: wrap the scaler inside the Pipeline passed to cross_val_score,
# so it's refit on each fold's training data only
correct_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

correct_scores = cross_val_score(correct_pipe, df[numeric_cols], y, cv=5, scoring='roc_auc')

print('Correct CV scores:', correct_scores)
print('Correct CV mean:  ', correct_scores.mean())
print('Difference in means (leaked - correct):', leaked_scores.mean() - correct_scores.mean())

Correct CV scores: [0.65971959 0.66148148 0.72888889 0.68074074 0.66111111]
Correct CV mean:   0.6783883625952591
Difference in means (leaked - correct): -0.0003014329221225909


**Explanation:** the difference here is essentially negligible (~0.0003 AUC) — on this dataset, three gentle, well-behaved numeric features and 600 rows don't give a global-fit scaler much room to diverge fold-by-fold from a per-fold-fit scaler. That said, the *risk* is real in general: the size of this leak scales with how much each fold's held-out data differs from the full dataset's statistics — smaller folds, noisier features, more outliers, or a `fit_transform` that's doing more than simple centering/scaling (e.g. target encoding, feature selection based on correlation with `y`) can all turn this same mistake into a much bigger, more misleading inflation of the reported CV score. Wrapping the scaler inside the `Pipeline` passed to `cross_val_score` is the correct habit regardless of how small the effect looks on any one dataset, because it guarantees the scaler is refit on only that fold's training data every time.